# UCI Heart Disease データセット（Cleveland）

出典：[UCI Machine Learning Repository — Heart Disease](https://archive.ics.uci.edu/dataset/45/heart+disease)（使用データ：`processed.cleveland.data`）

1. **age（年齢）**：年齢（年）
2. **sex（性別）**：`1`＝男性、`0`＝女性
3. **cp（胸痛の種類）**
    - `1`：典型的狭心症
    - `2`：非典型的狭心症
    - `3`：非狭心症性胸痛
    - `4`：無症状
4. **trestbps（安静時血圧）**：入院時の安静時血圧（mmHg）
5. **chol（血清コレステロール値）**：血清コレステロール値（mg/dL）
6. **fbs（空腹時血糖の閾値判定）**
    - `1`：空腹時血糖値が120 mg/dLを超える
    - `0`：空腹時血糖値が120 mg/dL以下
7. **restecg（安静時心電図所見）**
    - `0`：正常
    - `1`：ST-T波異常
    - `2`：Estesの基準による左室肥大の疑いまたは確定所見
8. **thalach（最大到達心拍数）**：最大到達心拍数
9. **exang（運動誘発性狭心症）**：`1`＝あり、`0`＝なし
10. **oldpeak（運動に伴うST低下度）**：安静時と比較した運動時のST低下量
11. **slope（最大運動時のSTセグメントの傾き）**
    - `1`：上昇型
    - `2`：水平型
    - `3`：下降型
12. **ca（造影で描出された主要血管の数）**：0～3
13. **thal（タリウム負荷心筋検査に関する所見）**
    - `3`：正常
    - `6`：固定性欠損
    - `7`：可逆性欠損
14. **num（心疾患の診断ラベル）**
    - `0`：心疾患なし
    - `1`～`4`：心疾患あり（この演習ではすべて`1`にまとめて二値分類）

## データの読み込み

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Clevelandデータの列名（説明変数13個＋目的変数num）
column_names = [
    'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs',
    'restecg', 'thalach', 'exang', 'oldpeak', 'slope',
    'ca', 'thal', 'num'
]

# UCIの加工済みClevelandデータを取得
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data'
df = pd.read_csv(url, header=None, names=column_names, na_values='?')
n_original = len(df)

# 欠損値を含む行を除外し、数値として読み込む
# （このデータではcaやthalに欠損値がある）
df = df.dropna().copy().reset_index(drop=True)
df = df.apply(pd.to_numeric, errors='raise')

# 目的変数を二値化：0＝疾患なし、1＝疾患あり（元の1～4をまとめる）
df['num'] = (df['num'] > 0).astype(int)

print(f'元データ: {n_original}例 / 欠損値による除外: {n_original - len(df)}例 / 解析対象: {len(df)}例')
print('クラスごとのデータ数:')
print(df['num'].value_counts().sort_index())

df.head()

## グラフによるデータの確認

In [ ]:
x_column = 'age'
y_column = 'trestbps'

fig, ax = plt.subplots()
ax.scatter(df.loc[df['num'] == 0, x_column], df.loc[df['num'] == 0, y_column],
           label='no heart disease', alpha=0.7)
ax.scatter(df.loc[df['num'] == 1, x_column], df.loc[df['num'] == 1, y_column],
           label='heart disease', alpha=0.7)
ax.legend()
ax.set_xlabel('age [years]')
ax.set_ylabel('resting blood pressure [mmHg]')
plt.show()

## データベースを学習用とテスト用に分割

In [ ]:
# 元の13個の説明変数を数値のまま使用する
X = df.drop(columns='num').to_numpy(dtype=float)
y = df['num'].to_numpy(dtype=int)

# 学習用（80%）とテスト用（20%）に分割
def train_test_split(X, y, test_size=0.2, random_state=None):
    rng = np.random.default_rng(random_state)
    indices = rng.permutation(len(y))
    n_test = int(len(y) * test_size)
    test_indices, train_indices = indices[:n_test], indices[n_test:]
    return X[train_indices], X[test_indices], y[train_indices], y[test_indices]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'学習用: {len(y_train)}例 / テスト用: {len(y_test)}例')

## フィッシャーの線形判別

In [ ]:
# 学習データだけを使って各クラスの平均ベクトルを求める
mean_0 = X_train[y_train == 0].mean(axis=0)
mean_1 = X_train[y_train == 1].mean(axis=0)
mean_diff = mean_0 - mean_1

# クラス内散布行列 S_w = S_0 + S_1
# 各S_cは、平均との差の外積の総和（共分散行列そのものではない）
S_w = np.zeros((X_train.shape[1], X_train.shape[1]))
for c, mean in ((0, mean_0), (1, mean_1)):
    centered = X_train[y_train == c] - mean
    S_w += centered.T @ centered

# フィッシャーの射影ベクトル w = S_w^(-1) (mean_0 - mean_1)
# 逆行列の明示計算より連立一次方程式の求解の方が数値的に適切
w = np.linalg.solve(S_w, mean_diff)

# 新しい1次元軸へ射影
X_train_lda = X_train @ w
X_test_lda = X_test @ w

# 閾値は学習データにおける射影後のクラス平均の中点
mean_lda_0 = X_train_lda[y_train == 0].mean()
mean_lda_1 = X_train_lda[y_train == 1].mean()
threshold = (mean_lda_0 + mean_lda_1) / 2

# wの向きを(mean_0 - mean_1)で定めたので、クラス0の射影平均はクラス1より大きい
# （S_wが正定値である場合）。閾値の両側でクラスを判定する
assert mean_lda_0 > mean_lda_1

y_pred = np.where(X_test_lda > threshold, 0, 1)

# 混同行列：行=真のクラス[0,1]、列=予測クラス[0,1]
def confusion_matrix(y_true, y_pred):
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tp = np.sum((y_true == 1) & (y_pred == 1))
    return np.array([[tn, fp],
                     [fn, tp]])

print('混同行列（行：真のクラス0,1／列：予測クラス0,1）:')
print(confusion_matrix(y_test, y_pred))

# 適合率（precision）、再現率（recall）、F1スコア、正解率（accuracy）を計算
def classification_report(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision_0 = tn / (tn + fn) if (tn + fn) else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) else 0.0
    precision_1 = tp / (tp + fp) if (tp + fp) else 0.0
    recall_1 = tp / (tp + fn) if (tp + fn) else 0.0
    f1_0 = 2 * precision_0 * recall_0 / (precision_0 + recall_0) if (precision_0 + recall_0) else 0.0
    f1_1 = 2 * precision_1 * recall_1 / (precision_1 + recall_1) if (precision_1 + recall_1) else 0.0
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    print('\n分類レポート（テストデータ）:')
    print(f'クラス 0 - 適合率: {precision_0:.2f}, 再現率: {recall_0:.2f}, F1スコア: {f1_0:.2f}')
    print(f'クラス 1 - 適合率: {precision_1:.2f}, 再現率: {recall_1:.2f}, F1スコア: {f1_1:.2f}')
    print(f'全体の正解率: {accuracy:.2f}')

classification_report(y_test, y_pred)

## 混同行列の見方

行が**実際のクラス**、列が**予測したクラス**（`0`：心疾患なし、`1`：心疾患あり）。

| 実際 ＼ 予測 | 0：心疾患なし | 1：心疾患あり |
|:--|:--:|:--:|
| **0：心疾患なし** | TN：真陰性（正しく「なし」と予測） | FP：偽陽性（誤って「あり」と予測） |
| **1：心疾患あり** | FN：偽陰性（誤って「なし」と予測） | TP：真陽性（正しく「あり」と予測） |